In [27]:
from dotenv import load_dotenv
from agents import Runner, Agent, trace
from openai import OpenAI
# from anthropic import Anthropic
import ast
import os
import json

In [2]:
load_dotenv(override=True)

True

# Code Learner / Understander


In [3]:
codeLearner = OpenAI(
    api_key=os.environ.get("ANTHROPIC_API_KEY"),  # Your Claude API key
    base_url="https://api.anthropic.com/v1/",  # the Claude API endpoint
)

In [4]:
system_message = """You are an experienced and helpful developer. 
                    You are specialized in understanding code writen by other developers and explaining to others.
                    Your first step would be identifying the language used by the user in the code.
                    You give a detailed explaination of the codes working and understanding of its use case.
                    This explaination will be sent to other agents to help them make changes in this code.
                    Please keep the explaination simple and easy to understand.
                    Your job is to analyze code and return structured JSON with:

                    - programming_language_used (Python, Jave, C++, Go, Kotlin, Javascript, etc.)
                    - problem_summary
                    - approach
                    - key_constructs
                    - complexity
                        - time
                        - space
                    - confidence (0 to 1)

                    Rules:
                    - Be concise
                    - Do NOT explain anything outside JSON
                    - Always return valid JSON

                    
                    -- IMPORTANT -- 
                    Response-format: 
                    {
                        "programming_language_used": ...,
                        "problem_summary" : ...,
                        "approach" : ...,
                        "key_constructs": ...,
                        "complexity": {
                            "time": ..,
                            "space": ..
                        },
                        "confidence": ...
                    }
"""

In [37]:
code = """
def twoSum(nums, target):
    for i in range(len(nums)):
        for j in range(i+1, len(nums)):
            if nums[i] + nums[j] == target:
                return [i, j]
"""

user_message = f"""
Analyze the following code:

Code:
{code}
"""

In [6]:
response = codeLearner.chat.completions.create(
    model="claude-sonnet-4-6",  # Claude model name
    messages=[
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_message},
    ],
)

In [7]:
print(response.choices[0].message.content)

```json
{
    "programming_language_used": "Python",
    "problem_summary": "Implements an LRU (Least Recently Used) Cache with a fixed capacity. It supports get and put operations, automatically evicting the least recently used item when the cache is full.",
    "approach": "Uses a dictionary (cache) for O(1) key-value storage and a list (order) to track the usage order of keys. The front of the list represents the least recently used item, and the end represents the most recently used. On every get or put, the accessed key is moved to the end of the list. When capacity is exceeded, the first element (LRU) is removed.",
    "key_constructs": [
        "Dictionary (self.cache): stores key-value pairs for fast lookup",
        "List (self.order): maintains insertion/access order of keys",
        "get(): retrieves value by key and marks it as recently used",
        "put(): inserts or updates a key-value pair, evicts LRU item if over capacity",
        "display(): prints all current cac

In [8]:
def clean_json_output(text):
    text = text.strip()
    
    # Remove markdown ```json or ```
    if text.startswith("```"):
        text = text.split("```")[1]  # remove first ```
        text = text.replace("json", "", 1)  # remove 'json' if present
        text = text.strip()
    
    if text.endswith("```"):
        text = text[:-3].strip()
    
    return text

In [9]:
output = json.loads(clean_json_output(response.choices[0].message.content))

In [10]:
output

{'programming_language_used': 'Python',
 'problem_summary': 'Implements an LRU (Least Recently Used) Cache with a fixed capacity. It supports get and put operations, automatically evicting the least recently used item when the cache is full.',
 'approach': 'Uses a dictionary (cache) for O(1) key-value storage and a list (order) to track the usage order of keys. The front of the list represents the least recently used item, and the end represents the most recently used. On every get or put, the accessed key is moved to the end of the list. When capacity is exceeded, the first element (LRU) is removed.',
 'key_constructs': ['Dictionary (self.cache): stores key-value pairs for fast lookup',
  'List (self.order): maintains insertion/access order of keys',
  'get(): retrieves value by key and marks it as recently used',
  'put(): inserts or updates a key-value pair, evicts LRU item if over capacity',
  'display(): prints all current cache entries in LRU to MRU order'],
 'complexity': {'time

In [11]:
output.get("key_constructs")

['Dictionary (self.cache): stores key-value pairs for fast lookup',
 'List (self.order): maintains insertion/access order of keys',
 'get(): retrieves value by key and marks it as recently used',
 'put(): inserts or updates a key-value pair, evicts LRU item if over capacity',
 'display(): prints all current cache entries in LRU to MRU order']

In [5]:
code = """
class LRUCache:
    def __init__(self, capacity):
        self.capacity = capacity
        self.cache = {}
        self.order = []

    def get(self, key):
        if key in self.cache:
            # Move key to end (most recently used)
            self.order.remove(key)
            self.order.append(key)
            return self.cache[key]
        return -1

    def put(self, key, value):
        if key in self.cache:
            self.cache[key] = value
            self.order.remove(key)
            self.order.append(key)
        else:
            if len(self.cache) >= self.capacity:
                # Remove least recently used
                lru = self.order[0]
                del self.cache[lru]
                self.order.pop(0)
            
            self.cache[key] = value
            self.order.append(key)

    def display(self):
        for k in self.order:
            print(k, self.cache[k])
"""

user_message = f"""
Analyze the following code:

Code:
{code}
"""

In [33]:
def run_code_understander(code):
    codeLearner = OpenAI(
        api_key=os.environ.get("ANTHROPIC_API_KEY"),  # Your Claude API key
        base_url="https://api.anthropic.com/v1/",  # the Claude API endpoint
    )
    system_message = """You are an experienced and helpful developer. 
                    You are specialized in understanding code writen by other developers and explaining to others.
                    Your first step would be identifying the language used by the user in the code.
                    You give a detailed explaination of the codes working and understanding of its use case.
                    This explaination will be sent to other agents to help them make changes in this code.
                    Please keep the explaination simple and easy to understand.
                    Your job is to analyze code and return structured JSON with:

                    - programming_language_used (Python, Jave, C++, Go, Kotlin, Javascript, etc.)
                    - problem_summary
                    - approach
                    - key_constructs
                    - complexity
                        - time
                        - space
                    - confidence (0 to 1)

                    Rules:
                    - Be concise
                    - Do NOT explain anything outside JSON
                    - Always return valid JSON

                    
                    -- IMPORTANT -- 
                    Response-format: 
                    {
                        "programming_language_used": ...,
                        "problem_summary" : ...,
                        "approach" : ...,
                        "key_constructs": ...,
                        "complexity": {
                            "time": ..,
                            "space": ..
                        },
                        "confidence": ...
                    }
    """
    
    user_message = f"""
        Analyze the following code:

        Code:
        {code}
    """

    response = codeLearner.chat.completions.create(
        model="claude-sonnet-4-6",  # Claude model name
        messages=[
            {"role": "system", "content": system_message},
            {"role": "user", "content": user_message},
        ],
    )

    return json.loads(clean_json_output(response.choices[0].message.content))

In [34]:
output = run_code_understander(code)

In [35]:
print(output)

{'programming_language_used': 'Python', 'problem_summary': 'Implementation of a Least Recently Used (LRU) Cache with a fixed capacity. It supports get and put operations, evicting the least recently used item when the cache is full.', 'approach': 'Uses a dictionary (cache) for O(1) key-value storage and a list (order) to track the usage order of keys. The front of the list represents the least recently used item, and the end represents the most recently used. On every get or put, the accessed key is moved to the end of the list. When capacity is exceeded, the first element (LRU) is evicted.', 'key_constructs': ['Class with __init__, get, put, and display methods', 'Dictionary (self.cache) for key-value storage', 'List (self.order) for tracking usage order', 'list.remove() to delete a key from its current position', 'list.append() to mark a key as most recently used', 'list.pop(0) to evict the least recently used key'], 'complexity': {'time': 'O(n) for both get and put due to list.remov

# Code Corrector / Recommender

In [30]:
def run_codeCorrector(code, understanding):
    codeCorrector = OpenAI(
        api_key=os.environ.get("ANTHROPIC_API_KEY"),  # Your Claude API key
        base_url="https://api.anthropic.com/v1/",  # the Claude API endpoint
    )
    
    system_message = """
        You are a senior software engineer and expert code reviewer.

        Your job is NOT to understand the code from scratch.
        You will be provided with:
        1. The original code
        2. A structured analysis from a previous agent

        Use this information to perform a deep technical review and improve the code.

        ---

        Your tasks:

        1. Validate correctness of the code
        2. Identify logical bugs and issues
        3. Detect missing edge case handling
        4. Analyze time and space complexity
        5. Suggest optimizations
        6. Recommend better tools, libraries, or frameworks (UNBIASED and practical)
        7. Provide a corrected version of the code fixing major/common issues
        (DO NOT completely rewrite unless necessary — keep structure similar)

        ---

        Return ONLY valid JSON.

        DO NOT:
        - Add explanations outside JSON
        - Wrap output in ```json or ```
        - Repeat input unnecessarily

        ---

        IMPORTANT FORMAT:

        {
        "correctness": "Correct | Partially Correct | Incorrect",
        "bugs": ["...", "..."],
        "edge_cases": ["...", "..."],
        "complexity": {
            "time": "...",
            "space": "..."
        },
        "optimizations": ["...", "..."],
        "improved_approach": "...",
        "tools_recommendation": [
            {
            "current": "...",
            "suggested": "...",
            "reason": "..."
            }
        ],
        "corrected_code": "...",
        "confidence": 0.0
        }

        ---

        STRICT RULES:

        - Be critical and precise (like a real interviewer)
        - Do NOT praise unnecessarily

        - "bugs" must include real issues (not generic)
        - "edge_cases" must be realistic and relevant
        - "optimizations" must be actionable

        - "corrected_code":
            - Fix major bugs and inefficiencies
            - Improve readability if needed
            - Keep original structure as much as possible
            - DO NOT over-engineer
            - Must be valid and runnable

        - Tool recommendations must be:
            - Relevant
            - Practical
            - Unbiased

        - If no tool improvement is needed, return []

        - Confidence must be between 0 and 1

        ---
    """

    user_message = f"""
        Here is the code understanding for the previous agent:\n
        {understanding}

        Code:
        {code}
    """

    response = codeCorrector.chat.completions.create(
        model="claude-opus-4-6",  # Claude model name
        messages=[
            {"role": "system", "content": system_message},
            {"role": "user", "content": user_message},
        ],
    )
    
    return json.loads(response.choices[0].message.content)

In [31]:
codeCorrector_output = run_codeCorrector(code, output)

In [32]:
print(codeCorrector_output)

{'correctness': 'Partially Correct', 'bugs': ['capacity of 0 or negative values is not handled — put() will still insert items into the cache since `len(self.cache) >= self.capacity` would be true but the item is added after eviction, resulting in a cache that never should have held anything', "list.remove(key) will raise ValueError if key appears multiple times or is somehow not in the list (shouldn't happen in normal flow, but there's no defensive handling)", 'self.order.pop(0) is O(n) due to shifting all elements — this is a performance bug for large caches'], 'edge_cases': ['capacity <= 0: The cache should reject all insertions, but currently it will store one item after evicting nothing (when capacity is 0) or store items freely (when capacity is negative)', 'Putting the same key repeatedly: Handled correctly — updates value and moves to end', 'Getting a non-existent key: Returns -1 correctly', 'Single capacity cache: Works but with O(n) overhead on every operation due to list ope

In [12]:
codeCorrector = OpenAI(
    api_key=os.environ.get("ANTHROPIC_API_KEY"),  # Your Claude API key
    base_url="https://api.anthropic.com/v1/",  # the Claude API endpoint
)

In [13]:
system_message = """
You are a senior software engineer and expert code reviewer.

Your job is NOT to understand the code from scratch.
You will be provided with:
1. The original code
2. A structured analysis from a previous agent

Use this information to perform a deep technical review and improve the code.

---

Your tasks:

1. Validate correctness of the code
2. Identify logical bugs and issues
3. Detect missing edge case handling
4. Analyze time and space complexity
5. Suggest optimizations
6. Recommend better tools, libraries, or frameworks (UNBIASED and practical)
7. Provide a corrected version of the code fixing major/common issues
   (DO NOT completely rewrite unless necessary — keep structure similar)

---

Return ONLY valid JSON.

DO NOT:
- Add explanations outside JSON
- Wrap output in ```json or ```
- Repeat input unnecessarily

---

IMPORTANT FORMAT:

{
  "correctness": "Correct | Partially Correct | Incorrect",
  "bugs": ["...", "..."],
  "edge_cases": ["...", "..."],
  "complexity": {
    "time": "...",
    "space": "..."
  },
  "optimizations": ["...", "..."],
  "improved_approach": "...",
  "tools_recommendation": [
    {
      "current": "...",
      "suggested": "...",
      "reason": "..."
    }
  ],
  "corrected_code": "...",
  "confidence": 0.0
}

---

STRICT RULES:

- Be critical and precise (like a real interviewer)
- Do NOT praise unnecessarily

- "bugs" must include real issues (not generic)
- "edge_cases" must be realistic and relevant
- "optimizations" must be actionable

- "corrected_code":
    - Fix major bugs and inefficiencies
    - Improve readability if needed
    - Keep original structure as much as possible
    - DO NOT over-engineer
    - Must be valid and runnable

- Tool recommendations must be:
    - Relevant
    - Practical
    - Unbiased

- If no tool improvement is needed, return []

- Confidence must be between 0 and 1

---
"""

In [14]:
user_message = f"""
Here is the code understanding for the previous agent:\n
{output}

Code:
{code}
"""

In [15]:
response = codeCorrector.chat.completions.create(
    model="claude-opus-4-6",  # Claude model name
    messages=[
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_message},
    ],
)

In [16]:
print(response.choices[0].message.content)



{
  "correctness": "Partially Correct",
  "bugs": [
    "capacity of 0 or negative values is not handled — put() will still insert items since `len(self.cache) >= self.capacity` would be True but the item is still added after eviction, leading to a cache that should hold 0 items but holds 1",
    "No validation that capacity is a positive integer; passing None, a float, or a string will cause runtime errors at comparison"
  ],
  "edge_cases": [
    "capacity = 0: put() evicts the only item via order[0] on the second insert but allows the first insert, resulting in a cache that should be empty but holds one item",
    "capacity = 1: works correctly but worth verifying eviction on every new key",
    "Duplicate put() with same key and same value: functionally correct but does unnecessary work",
    "get() on an empty cache: returns -1 correctly",
    "Very large number of operations: O(n) list.remove() becomes a performance bottleneck",
    "Non-hashable keys (e.g., lists): will raise 

In [17]:
codeCorrector_output = json.loads(response.choices[0].message.content)

In [62]:
print(codeCorrector_output.get("corrected_code"))

from collections import OrderedDict


class LRUCache:
    def __init__(self, capacity):
        if capacity <= 0:
            raise ValueError("Capacity must be a positive integer")
        self.capacity = capacity
        self.cache = OrderedDict()

    def get(self, key):
        if key in self.cache:
            # Move key to end (most recently used)
            self.cache.move_to_end(key)
            return self.cache[key]
        return -1

    def put(self, key, value):
        if key in self.cache:
            self.cache[key] = value
            self.cache.move_to_end(key)
        else:
            if len(self.cache) >= self.capacity:
                # Remove least recently used (first item)
                self.cache.popitem(last=False)
            self.cache[key] = value

    def display(self):
        for k, v in self.cache.items():
            print(k, v)



## Reviewer Agent

In [18]:
system_message = """
You are a senior software engineer performing a professional code review.

Your job is to evaluate code quality, maintainability, and production readiness.

You will be provided with:
1. Original code
2. Code understanding analysis
3. Technical review from another agent

---

Your tasks:

1. Evaluate readability and clarity
2. Identify code quality issues
3. Assess maintainability and scalability
4. Check adherence to best practices
5. Suggest improvements for production-level code

---

Return ONLY valid JSON.

DO NOT:
- Add explanations outside JSON
- Wrap output in ```json or ```
- Repeat input unnecessarily

---

IMPORTANT FORMAT:

{
  "readability_score": 0,
  "code_quality_issues": ["...", "..."],
  "maintainability_issues": ["...", "..."],
  "best_practice_violations": ["...", "..."],
  "strengths": ["...", "..."],
  "improvement_suggestions": ["...", "..."],
  "production_readiness": {
    "status": "Low | Medium | High",
    "issues": ["...", "..."]
  },
  "final_summary": "...",
  "confidence": 0.0
}

---

STRICT RULES:

- readability_score: 0-10

- Be critical and realistic (like a senior engineer in code review)
- Do NOT be overly polite

- "code_quality_issues" must be specific
- "maintainability_issues" must reflect long-term concerns
- "best_practice_violations" must be actionable

- "production_readiness":
    - Evaluate if code can be used in real-world systems
    - Consider error handling, scalability, robustness

- "final_summary" should be concise (2-3 lines)

- Confidence must be between 0 and 1

---
"""

In [19]:
code_quality_agent = Agent(
    name="Code Quality Reviewer",
    instructions=system_message,
    model="gpt-5-mini"
)

In [28]:
async def run_code_quality_agent(code, understanding, technical_review):

    input_text = f"""
Code:
{code}

Understanding:
{understanding}

Technical Review:
{technical_review}
"""
    with trace("CodeQualityAgent"):    
        result = await Runner.run(
            code_quality_agent,
            input=input_text
        )

    raw = result.final_output
    # cleaned = extract_json(raw)

    try:
        return json.loads(raw)
    except:
        print("❌ Parsing failed")
        print(raw)
        return None

In [29]:
final_output = await run_code_quality_agent(code, output, codeCorrector_output)

In [24]:
final_output

{'readability_score': 6,
 'code_quality_issues': ['Algorithmic inefficiency: get() and put() use list.remove() and list.pop(0) which are O(n) operations — unacceptable for large caches.',
  'No input validation for capacity: __init__ accepts non-integer or non-positive values; capacity <= 0 will cause IndexError at eviction (order[0] on empty list).',
  'display() prints directly to stdout instead of returning structured data — hard to test and inappropriate for library code.',
  'Inconsistent API semantics: get() returns -1 for missing keys (magic value) instead of None or raising KeyError; not documented.',
  'Duplicate logic for moving keys to MRU in get() and put() — violates DRY and increases surface area for bugs.',
  'Internal attributes are public (self.cache, self.order) and not documented; breaks encapsulation and can be accidentally mutated.',
  'No error handling for non-hashable keys (dict will raise TypeError) or invalid key/value types — no defensive checks or clear cont

In [38]:
async def run_full_code_review(code):

    print("🟡 Running Code Understanding Agent...")
    understanding = run_code_understander(code) 
    
    print("✅ Understanding Done\n")

    print("🔵 Running Claude Review Agent...")
    technical_review = run_codeCorrector(code, understanding)
    
    print("✅ Technical Review Done\n")

    print("🟢 Running Code Quality Agent...")
    quality_review = await run_code_quality_agent(code, understanding, technical_review)
    
    print("✅ Quality Review Done\n")

    final_output = {
        "understanding": understanding,
        "technical_review": technical_review,
        "quality_review": quality_review
    }

    return final_output

In [39]:
result = await run_full_code_review(code)

🟡 Running Code Understanding Agent...
✅ Understanding Done

🔵 Running Claude Review Agent...
✅ Technical Review Done

🟢 Running Code Quality Agent...
✅ Quality Review Done



# Fully Optimized Code with Validation Loop

In [54]:
import re
def extract_json(text):
    match = re.search(r"\{.*\}", text, re.DOTALL)
    return match.group(0) if match else text

In [61]:
def call_claude_optimizer(user_message):
  system_prompt = """
    You are an expert software engineer specializing in code optimization and production-level improvements.

    Your role is to generate a fully optimized and production-ready version of the given code using the feedback and reviews.

    You have to rewrite the code and return it.
    ---

    Responsibilities:
    - Ensures to make changes according to the feedback and reviews provided by the agents first
    - Improve time and space complexity where possible
    - Use optimal data structures and algorithms
    - Ensure code is clean, readable, and maintainable
    - Apply best practices
    - Ensure correctness

    ---

    Output Requirements:
    - Return ONLY valid JSON
    - Return the updated code in optimized_code
    - Do NOT include any text outside JSON
    - Do NOT wrap output in ``` or ```json

    ---

    FORMAT:

    {
      "optimized_code": "...",
      "changes_made": ["...", "..."],
      "optimization_summary": "..."
    }
    """
  codeCorrector = OpenAI(
        api_key=os.environ.get("ANTHROPIC_API_KEY"),  # Your Claude API key
        base_url="https://api.anthropic.com/v1/",  # the Claude API endpoint
  )
  response = codeCorrector.chat.completions.create(
        model="claude-opus-4-6",  # Claude model name
        messages=[
            {"role": "system", "content": system_message},
            {"role": "user", "content": user_message},
        ],
    )
    
  return response.choices[0].message.content
  


async def optimize_with_validation(code, intial_result, max_iters=3):

    feedback = None

    for i in range(max_iters):

      print(f"\n🔵 Claude Optimization Iteration {i+1}")

      user_prompt = f"""
        Optimize this code to production level and according to the following feedbacks and Reviews.

        Understanding:
        {intial_result.get("understanding")}

        Technical Review:
        {intial_result.get("technical_review")}

        Quality Review:
        {intial_result.get("quality_review")}

        Previous Feedback (if any):
        {feedback}

        Code:
        {code}
        """
      claude_output = call_claude_optimizer(user_prompt)
      claude_json = json.loads(extract_json(claude_output))
      print("RAW CLAUDE OUTPUT:\n", claude_output)
      print("\n \n Claude Json: \n", claude_json)

      optimized_code = claude_json["optimized_code"]

      print("🟢 GPT Validation...")

      gpt_prompt = f"""
        Check if the optimized code satisfies:
        - correctness
        - improvements suggested earlier
        - is runnable and logically valid

        Return JSON:
        {{
        "valid": true/false,
        "issues": ["..."],
        "feedback": "..."
        }}

        Code:
        {optimized_code}
      """

      gpt_result = await Runner.run(gpt_reviewer_agent, input=gpt_prompt, model="gpt-5-mini")
      gpt_json = json.loads(extract_json(gpt_result.final_output))

      if gpt_json["valid"]:
          print("✅ Optimization Successful")
          return optimized_code

      print("❌ Needs Improvement:", gpt_json["issues"])

      feedback = gpt_json["feedback"]
      code = optimized_code  # iterative refinement

    print("⚠️ Max iterations reached")
    return optimized_code

In [56]:
code = """
class LRUCache:
    def __init__(self, capacity):
        self.capacity = capacity
        self.cache = {}
        self.order = []

    def get(self, key):
        if key in self.cache:
            self.order.remove(key)
            self.order.append(key)
            return self.cache[key]
        return -1

    def put(self, key, value):
        if key in self.cache:
            self.cache[key] = value
            self.order.remove(key)
            self.order.append(key)
        else:
            if len(self.cache) >= self.capacity:
                lru = self.order[0]
                del self.cache[lru]
                self.order.pop(0)
            
            self.cache[key] = value
            self.order.append(key)
"""

inital_output = await run_full_code_review(code)

optimized_code = await optimize_with_validation(code, inital_output)

print(optimized_code)

🟡 Running Code Understanding Agent...
✅ Understanding Done

🔵 Running Claude Review Agent...
✅ Technical Review Done

🟢 Running Code Quality Agent...
✅ Quality Review Done


🔵 Claude Optimization Iteration 1


KeyError: 'optimized_code'

In [63]:
optimized_code_output = await optimize_with_validation(code, inital_output)


🔵 Claude Optimization Iteration 1
RAW CLAUDE OUTPUT:
 

{
  "readability_score": 6,
  "code_quality_issues": [
    "get() and put() both call list.remove(key) which is O(n) — fundamentally breaks the O(1) contract expected of an LRU cache.",
    "list.pop(0) is O(n) due to element shifting; combined with list.remove(), every operation is linear in the number of cached items.",
    "No input validation on capacity: capacity <= 0 causes IndexError on self.order[0] when eviction is triggered on an empty list, or allows unbounded growth for negative values.",
    "Returning -1 as a sentinel for missing keys is ambiguous — -1 could be a legitimate cached value. No documentation clarifies this contract.",
    "Duplicate remove/append pattern in both get() and put() violates DRY and increases risk of desynchronization between cache and order.",
    "No handling of non-hashable keys — dict operations will raise TypeError with no meaningful error message.",
    "Not thread-safe: concurrent get

KeyError: 'optimized_code'

# Chat Refinement with Validation Loop

In [46]:
def call_claude_chat(user_message):
  system_message = """
    You are an expert software engineer helping refine and modify code based on user requests.

    ---

    Responsibilities:
    - Apply the requested changes to the code
    - Always remember to keep the context and changes provided by agents in reviews and feedbacks
    - Maintain correctness
    - Keep code clean and readable
    - Ensure the code remains functional

    ---

    Output Requirements:
    - Return ONLY valid JSON
    - Do NOT include text outside JSON
    - Do NOT wrap output in ``` or ```json

    ---

    FORMAT:

    {
      "updated_code": "...",
      "changes_made": ["...", "..."],
      "explanation": "..."
    }
    """
  codeCorrector = OpenAI(
        api_key=os.environ.get("ANTHROPIC_API_KEY"),  # Your Claude API key
        base_url="https://api.anthropic.com/v1/",  # the Claude API endpoint
  )
  response = codeCorrector.chat.completions.create(
        model="claude-opus-4-6",  # Claude model name
        messages=[
            {"role": "system", "content": system_message},
            {"role": "user", "content": user_message},
        ],
    )
    
  return response.choices[0].message.content

async def chat_with_validation(code, intial_result, user_query, max_iters=3):

    feedback = None

    for i in range(max_iters):
    
      print(f"\n🔵 Claude Chat Iteration {i+1}")

      user_prompt = f"""
        Modify the code based on user request.

        User Request:
        {user_query}

        Understanding:
        {intial_result.get("understanding")}

        Technical Review:
        {intial_result.get("technical_review")}

        Previous Feedback (if any):
        {feedback}

        Code:
        {code}
      """

      claude_output = call_claude_chat(user_prompt)
      claude_json = json.loads(extract_json(claude_output))

      updated_code = claude_json["updated_code"]

      print("🟢 GPT Validation...")

      gpt_prompt = f"""
        Check:
        - Is this code correct?
        - Is it runnable?
        - Does it satisfy the user request?

        Return JSON:
        {{
        "valid": true/false,
        "issues": ["..."],
        "feedback": "..."
        }}

        Code:
        {updated_code}
        """

      gpt_result = await Runner.run(gpt_reviewer_agent, input=gpt_prompt)
      gpt_json = json.loads(extract_json(gpt_result.final_output))

      if gpt_json["valid"]:
          print("✅ Chat Update Successful")
          return updated_code

      print("❌ Needs Fix:", gpt_json["issues"])

      feedback = gpt_json["feedback"]
      code = updated_code

    print("⚠️ Max iterations reached")
    return updated_code